# Depth scan — how far QAOA gets on `ibm_fez` as a function of $p$

**Dr. Muhammad Faryad** · follow-up to `NB16a–d`

In the main study the circuit depth grew together with the instance
($p = 2, 3, 4, 5$ for $n = 6, 8, 10, 12$), so "the device does worse at larger $n$" and "the
circuit got deeper" cannot be separated. This notebook separates them.

The trick is that the optimiser is not needed. For $n \le 12$ the noiseless QAOA landscape can be
optimised **exactly and classically**, so for each $(n, p)$ we can compute the best parameters a
perfect optimiser could ever return at that depth, and then simply *measure* what the device makes
of them — one job, 8,192 shots. What comes back is the depth–noise trade-off with the optimiser
removed from the picture:

* the noiseless approximation ratio rises monotonically with $p$ (more layers, more expressive),
* the measured one rises, peaks, and falls, because $3p\binom{n}{2}$ two-qubit gates cost more
  fidelity than the extra layer buys.

For $n = 12$ a second parameter set is measured at every depth: the point that maximises CVaR under
the **depolarising model fitted to the first study**,
$\eta = 1 - \exp[-(0.191 + 0.00135\,N_{2q})]$. Comparing the two curves says whether a
noise-aware optimiser could recover any of the loss, or whether the depth itself is the ceiling.

Cost: 4 sizes × 5 depths + 5 extra at $n = 12$ = **25 jobs ≈ 2.5 minutes of metered QPU**.
The classical pre-optimisation takes a couple of minutes of laptop CPU before anything is
submitted. The JSON is written after every size.

In [2]:
import json, time, warnings
from math import comb

import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

warnings.filterwarnings("ignore")

ACCOUNT_NAME = "Faryad-free-ibm"
BACKEND_NAME = "ibm_fez"

SIZES  = [6, 8, 10, 12]
DEPTHS = [1, 2, 3, 4, 5]
ALPHA  = 0.5
SHOTS_FINAL = 8192
NOISE_AWARE_AT = 12            # also measure model-optimal parameters at this size
QPU_BUDGET_S = 400             # model predicts ~157 s
N_STATES_CACHE = {n: 2 ** n for n in SIZES}

# depolarising law fitted to the four main runs (see the paper)
ETA_A, ETA_B = 0.1912, 0.001350
eta_of = lambda n2q: 1 - np.exp(-(ETA_A + ETA_B * n2q))
print("predicted eta:", {(n, p): round(float(eta_of(3 * p * n * (n - 1) // 2)), 3)
                         for n in SIZES for p in (1, 5)})

predicted eta: {(6, 1): 0.223, (6, 5): 0.39, (8, 1): 0.263, (8, 5): 0.531, (10, 1): 0.312, (10, 5): 0.668, (12, 1): 0.368, (12, 5): 0.783}


In [3]:
asset_names = ['Google', 'Intel', 'Nvidia', 'Goldman Sachs', 'ExxonMobil', 'IBM',
               'JPMorgan', 'Bank of America', 'Johnson & Johnson', 'Pfizer',
               'Walmart', 'Amazon']
tickers = ['GOOG', 'INTC', 'NVDA', 'GS', 'XOM', 'IBM', 'JPM', 'BAC', 'JNJ', 'PFE', 'WMT', 'AMZN']

expected_returns_12 = np.array(
    [59.14, 168.17, 28.48, 41.97, 48.07, 12.59, 24.56, 31.27, 46.84, 20.29, 14.46, 23.06])

covariance_12 = np.array(
    [[10.41, 4.95, 2.92, 3.04, -2.16, 1.20, 1.36, 1.29, -0.16, 0.51, 0.21, 5.63],
     [4.95, 62.12, 10.62, 8.27, -3.19, -2.50, 1.97, 1.80, -2.81, -0.11, -1.29, 4.88],
     [2.92, 10.62, 13.48, 5.41, -1.81, 0.12, 1.52, 1.06, -1.55, -0.51, -1.67, 3.69],
     [3.04, 8.27, 5.41, 9.88, -1.54, 0.52, 4.54, 3.72, -0.88, 0.41, -0.84, 2.51],
     [-2.16, -3.19, -1.81, -1.54, 6.51, -0.71, -0.36, -0.43, 0.52, -0.18, 0.41, -1.97],
     [1.20, -2.50, 0.12, 0.52, -0.71, 23.34, 1.66, 2.14, 0.15, 1.70, -1.17, 2.05],
     [1.36, 1.97, 1.52, 4.54, -0.36, 1.66, 4.97, 3.57, 0.09, 0.56, -0.28, 1.27],
     [1.29, 1.80, 1.06, 3.72, -0.43, 2.14, 3.57, 4.50, -0.13, 0.50, 0.08, 1.32],
     [-0.16, -2.81, -1.55, -0.88, 0.52, 0.15, 0.09, -0.13, 3.44, 1.29, 1.37, -0.94],
     [0.51, -0.11, -0.51, 0.41, -0.18, 1.70, 0.56, 0.50, 1.29, 5.58, 0.23, 0.54],
     [0.21, -1.29, -1.67, -0.84, 0.41, -1.17, -0.28, 0.08, 1.37, 0.23, 5.98, -0.89],
     [5.63, 4.88, 3.69, 2.51, -1.97, 2.05, 1.27, 1.32, -0.94, 0.54, -0.89, 11.84]])

LAMBDAS = {6: 1.771087, 8: 1.652797, 10: 1.553166, 12: 1.360032}   # as in the four main runs
MUS     = {6: 8.0,      8: 9.0,      10: 12.0,     12: 13.0}


def build_instance(n):
    """Everything classical about the n-asset instance, exactly as in the main runs."""
    B, lam, mu = n // 2, LAMBDAS[n], MUS[n]
    r = expected_returns_12[:n]
    C = covariance_12[:n, :n]
    N = 2 ** n
    bits = (np.arange(N)[:, None] >> np.arange(n)[None, :]) & 1
    cost = (lam * np.einsum("ki,ij,kj->k", bits, C, bits) - bits @ r
            + mu * (bits.sum(1) - B) ** 2)
    feas = bits.sum(1) == B

    Q = lam * C + mu * np.ones((n, n))
    L = r + 2 * mu * B * np.ones(n)
    offset = mu * B ** 2
    h = np.zeros(n)
    J = np.zeros((n, n))
    for i in range(n):
        offset += Q[i, i] / 2 - L[i] / 2
        h[i] += -Q[i, i] / 2 + L[i] / 2
    for i in range(n):
        for j in range(i + 1, n):
            cij = 2 * Q[i, j]
            offset += cij / 4
            h[i] -= cij / 4
            h[j] -= cij / 4
            J[i, j] += cij / 4
    scale = max(np.abs(h).max(), np.abs(J).max())
    h, J = h / scale, J / scale
    z = 1 - 2 * bits
    diag = z @ h
    for i in range(n):
        for j in range(i + 1, n):
            diag += J[i, j] * z[:, i] * z[:, j]
    assert np.abs(cost - (scale * diag + offset)).max() < 1e-8, "Ising mapping is wrong"

    return dict(n=n, B=B, lam=lam, mu=mu, names=[tickers[i] for i in range(n)],
                r=r, C=C, N=N, bits=bits, cost=cost, feas=feas, h=h, J=J,
                diag=diag, scale=float(scale), offset=float(offset),
                order=np.argsort(diag), C_opt=float(cost.min()),
                opt_index=int(cost.argmin()), C_worst=float(cost[feas].max()),
                C_mean_feas=float(cost[feas].mean()))


def qaoa_probs(inst, params, reps):
    """Noiseless QAOA distribution; params = [beta_0..., gamma_0...]."""
    n, N, diag = inst["n"], inst["N"], inst["diag"]
    betas, gammas = params[:reps], params[reps:]
    psi = np.ones(N, dtype=complex) / np.sqrt(N)
    for k in range(reps):
        psi = psi * np.exp(-1j * gammas[k] * diag)
        psi = psi.reshape([2] * n)
        cb, sb = np.cos(betas[k]), -1j * np.sin(betas[k])
        for q in range(n):
            ax = n - 1 - q
            psi = np.moveaxis(psi, ax, 0)
            a, b = psi[0].copy(), psi[1].copy()
            psi[0], psi[1] = cb * a + sb * b, sb * a + cb * b
            psi = np.moveaxis(psi, 0, ax)
        psi = psi.reshape(-1)
    return np.abs(psi) ** 2


def cvar(inst, probs, alpha):
    o = inst["order"]
    e, p = inst["diag"][o], probs[o]
    cum = np.cumsum(p)
    k = min(int(np.searchsorted(cum, alpha)) + 1, p.size)
    w = p[:k].copy()
    w[-1] -= max(cum[k - 1] - alpha, 0.0)
    return float(e[:k] @ w / w.sum()) if w.sum() > 0 else float(e[0])


def quality(inst, probs):
    feas, cost = inst["feas"], inst["cost"]
    fm = float(probs[feas].sum())
    cond = probs * feas / max(fm, 1e-15)
    mcf = float(cost @ cond)
    return dict(feas_mass=fm, mean_cost=float(cost @ probs), mean_cost_feas=mcf,
                ratio=float((inst["C_worst"] - mcf) / (inst["C_worst"] - inst["C_opt"])),
                p_opt=float(probs[inst["opt_index"]]),
                p_opt_feas=float(cond[inst["opt_index"]]))


def qaoa_swap_network(inst, reps):
    """QAOA on a linear chain; complete ZZ graph via an odd-even transposition network."""
    h, J, n = inst["h"], inst["J"], inst["n"]
    betas, gammas = ParameterVector("b", reps), ParameterVector("g", reps)
    qc = QuantumCircuit(QuantumRegister(n, "q"), ClassicalRegister(n, "c"))
    qc.h(range(n))
    perm = list(range(n))
    for layer in range(reps):
        g, be = gammas[layer], betas[layer]
        for pos in range(n):
            if h[perm[pos]]:
                qc.rz(2 * g * h[perm[pos]], pos)
        for step in range(n):
            for pos in range(step % 2, n - 1, 2):
                a, b = perm[pos], perm[pos + 1]
                w = J[min(a, b), max(a, b)]
                qc.cx(pos, pos + 1)                 # RZZ(2 g w) . SWAP in 3 CX
                qc.rz(2 * g * w, pos + 1)
                qc.cx(pos + 1, pos)
                qc.cx(pos, pos + 1)
                perm[pos], perm[pos + 1] = b, a
        qc.rx(2 * be, range(n))
    for pos in range(n):
        qc.measure(pos, perm[pos])                  # clbit index == logical qubit index
    return qc, perm


def best_chain(backend, n, beam=3000):
    """Lowest-total-error simple path of n physical qubits."""
    t = backend.target
    cz = {e: (p.error if p and p.error is not None else 0.01) for e, p in t["cz"].items()}
    ro = {q[0]: (p.error if p and p.error is not None else 0.02) for q, p in t["measure"].items()}
    adj = {}
    for (a, b), e in cz.items():
        adj.setdefault(a, []).append((b, e))
    w = lambda e: -np.log(max(1e-6, 1 - e))
    paths = [([q], w(ro.get(q, 0.02))) for q in range(backend.num_qubits)]
    for _ in range(n - 1):
        nxt = []
        for path, c in paths:
            for b, e in adj.get(path[-1], []):
                if b not in path:
                    nxt.append((path + [b], c + w(e) + w(ro.get(b, 0.02))))
        nxt.sort(key=lambda t: t[1])
        seen, keep = set(), []
        for path, c in nxt:
            key = (path[-1], frozenset(path))
            if key not in seen:
                seen.add(key)
                keep.append((path, c))
            if len(keep) >= beam:
                break
        paths = keep
    return min(paths, key=lambda t: t[1])

---
## Classical pre-optimisation

Multi-start Powell on the exact noiseless landscape, then the same again against the fitted depolarising model. Nothing here touches the QPU; it just decides which 25 parameter vectors are worth a job.

In [4]:
def readout_mix(inst, p, e):
    n = inst["n"]
    p = p.reshape([2] * n).copy()
    for q in range(n):
        ax = n - 1 - q
        p = np.moveaxis(p, ax, 0)
        a, b = p[0].copy(), p[1].copy()
        p[0], p[1] = (1 - e) * a + e * b, e * a + (1 - e) * b
        p = np.moveaxis(p, 0, ax)
    return p.reshape(-1)


def optimise(inst, reps, eta=0.0, e_ro=0.0, starts=10, seed=0):
    """Best parameters at depth `reps`, either noiseless (eta=0) or under the fitted model."""
    rng = np.random.default_rng(seed)
    best = (np.inf, None)
    for s in range(starts):
        k = np.arange(1, reps + 1)
        dt = rng.uniform(0.25, 1.1)
        x0 = np.concatenate([dt * (1 - (k - .5) / reps), -dt * k / reps])
        x0 = x0 + rng.uniform(-0.03, 0.03, 2 * reps)

        def f(x):
            pr = qaoa_probs(inst, x, reps)
            if eta > 0:
                pr = readout_mix(inst, (1 - eta) * pr + eta / inst["N"], e_ro)
            return cvar(inst, pr, ALPHA)

        r = minimize(f, x0, method="Powell",
                     options=dict(maxfev=500, xtol=1e-3, ftol=1e-4))
        if r.fun < best[0]:
            best = (float(r.fun), r.x.copy())
    return best[1]


service = QiskitRuntimeService(name=ACCOUNT_NAME)
backend = service.backend(BACKEND_NAME)
print(f"{backend.name}: {backend.status().pending_jobs} jobs queued")

RO = {q[0]: (p.error if p and p.error is not None else 0.02)
      for q, p in backend.target["measure"].items()}

plan = {}
t0 = time.time()
for n in SIZES:
    inst = build_instance(n)
    chain, _ = best_chain(backend, n)
    e_ro = float(np.mean([RO[q] for q in chain]))
    plan[n] = dict(inst=inst, chain=chain, e_ro=e_ro, params={})
    for p in DEPTHS:
        n2q = 3 * p * n * (n - 1) // 2
        x_ideal = optimise(inst, p, seed=n * 10 + p)
        entry = dict(noiseless=x_ideal.tolist(), n2q=n2q, eta=float(eta_of(n2q)))
        if n == NOISE_AWARE_AT:
            entry["model"] = optimise(inst, p, eta=eta_of(n2q), e_ro=e_ro,
                                      seed=100 + n * 10 + p).tolist()
        plan[n]["params"][p] = entry
        q = quality(inst, qaoa_probs(inst, x_ideal, p))
        print(f"n={n:2d} p={p}  {n2q:4d} 2q gates  eta_pred {entry['eta']:.3f}  "
              f"noiseless ratio {q['ratio']:.3f}  feas {q['feas_mass']:.3f}")
print(f"classical pre-optimisation took {time.time()-t0:.0f} s")

ibm_fez: 1 jobs queued
n= 6 p=1    45 2q gates  eta_pred 0.223  noiseless ratio 0.788  feas 0.428
n= 6 p=2    90 2q gates  eta_pred 0.269  noiseless ratio 0.930  feas 0.642
n= 6 p=3   135 2q gates  eta_pred 0.312  noiseless ratio 0.952  feas 0.696
n= 6 p=4   180 2q gates  eta_pred 0.352  noiseless ratio 0.943  feas 0.604
n= 6 p=5   225 2q gates  eta_pred 0.390  noiseless ratio 0.963  feas 0.591
n= 8 p=1    84 2q gates  eta_pred 0.263  noiseless ratio 0.692  feas 0.411
n= 8 p=2   168 2q gates  eta_pred 0.342  noiseless ratio 0.885  feas 0.567
n= 8 p=3   252 2q gates  eta_pred 0.412  noiseless ratio 0.932  feas 0.706
n= 8 p=4   336 2q gates  eta_pred 0.475  noiseless ratio 0.959  feas 0.759
n= 8 p=5   420 2q gates  eta_pred 0.531  noiseless ratio 0.968  feas 0.772
n=10 p=1   135 2q gates  eta_pred 0.312  noiseless ratio 0.657  feas 0.324
n=10 p=2   270 2q gates  eta_pred 0.426  noiseless ratio 0.902  feas 0.453
n=10 p=3   405 2q gates  eta_pred 0.522  noiseless ratio 0.933  feas 0.609
n=

In [5]:
# ============================================================
# The cost model, calibrated on the 102 jobs of the first study
#     qpu_seconds(job) = 3.00 s + (352.6 + 0.267 * D2q) us * shots
# reproduces every job of the four main runs to better than 0.05 s.
# ============================================================
JOB_FIXED_S = 3.00
T_SHOT = lambda d2q: 1e-6 * (352.6 + 0.2673 * d2q)

n_jobs_used, shots_used, qpu_used = 0, 0, 0.0
job_log = []


def predict_qpu(shots, d2q):
    return JOB_FIXED_S + T_SHOT(d2q) * shots


class BudgetStop(Exception):
    """Raised instead of overrunning QPU_BUDGET_S, so the notebook always saves its data."""


def submit(isa, params, shots, reps, n_states, d2q, tag, reserve=0.0):
    """ONE job, ONE PUB, len(params) parameter rows."""
    global n_jobs_used, shots_used, qpu_used
    params = np.atleast_2d(np.asarray(params, float))
    rows, total = params.shape[0], params.shape[0] * shots
    predicted = predict_qpu(total, d2q)
    if qpu_used + predicted + reserve > QPU_BUDGET_S:
        raise BudgetStop(f"'{tag}' would need ~{predicted:.1f} s on top of {qpu_used:.1f} s "
                         f"(+{reserve:.1f} s reserved); cap is {QPU_BUDGET_S:.0f} s")
    t0 = time.time()
    pub = np.zeros_like(params)
    order = [p.name for p in isa.parameters]
    for i, nm in enumerate([f"b[{k}]" for k in range(reps)] + [f"g[{k}]" for k in range(reps)]):
        pub[:, order.index(nm)] = params[:, i]
    job = sampler.run([(isa, pub)], shots=shots)
    data = job.result()[0].data.c
    dists = []
    for k in range(rows):                       # never call get_int_counts() unindexed
        cnt = data[k].get_int_counts()
        d_ = np.zeros(n_states)
        d_[list(cnt.keys())] = np.array(list(cnt.values())) / shots
        dists.append(d_)
    try:
        measured = float(job.metrics()["usage"]["quantum_seconds"])
    except Exception:
        measured = predicted
    n_jobs_used += 1
    shots_used += total
    qpu_used += measured
    job_log.append(dict(tag=tag, job=n_jobs_used, id=job.job_id(), rows=rows, shots=shots,
                        reps=reps, d2q=d2q, qpu_predicted=predicted, qpu_measured=measured,
                        wall=time.time() - t0, params=params.tolist(),
                        counts=[{int(i): int(round(x[i] * shots)) for i in np.nonzero(x)[0]}
                                for x in dists]))
    print(f"    job {n_jobs_used:3d} [{tag}] id={job.job_id()}  {measured:5.2f} qpu s "
          f"(model {predicted:5.2f})  {time.time()-t0:5.0f} s wall  "
          f"(total {qpu_used:6.1f} / {QPU_BUDGET_S:.0f} s)")
    return dists

---
## Measure

One job per $(n, p)$, plus the noise-aware set at $n = 12$. Each job is a single PUB with one parameter row and 8,192 shots.

In [6]:
sampler = SamplerV2(mode=backend)
sampler.options.dynamical_decoupling.enable = True
sampler.options.twirling.enable_gates = True
sampler.options.twirling.enable_measure = True

OUT = f"qaoa_jobs_depth_scan_{BACKEND_NAME}.json"
scan, stopped = {}, False

for n in SIZES:
    inst, chain = plan[n]["inst"], plan[n]["chain"]
    scan[n] = dict(chain=list(map(int, chain)), e_ro=plan[n]["e_ro"], depths={})
    for p in DEPTHS:
        circuit, perm = qaoa_swap_network(inst, p)
        pm = generate_preset_pass_manager(optimization_level=1, backend=backend,
                                          initial_layout=chain, seed_transpiler=42)
        isa = pm.run(circuit)
        used = sorted({isa.find_bit(q).index for i in isa.data for q in i.qubits
                       if i.operation.name in ("cz", "measure")})
        assert set(used) <= set(chain), "the pass manager moved the layout"
        d2q = isa.depth(lambda i: i.operation.name == "cz")
        n2q = sum(1 for i in isa.data if i.operation.name == "cz")

        entry = dict(n2q=n2q, d2q=d2q, eta_pred=plan[n]["params"][p]["eta"], meas={})
        for kind in ("noiseless", "model"):
            x = plan[n]["params"][p].get(kind)
            if x is None or stopped:
                continue
            try:
                d = submit(isa, np.atleast_2d(x), SHOTS_FINAL, p, inst["N"], d2q,
                           f"n{n}p{p}_{kind}", 0.0)[0]
            except BudgetStop as e:
                print(f"  budget stop: {e}")
                stopped = True
                continue
            entry["meas"][kind] = dict(
                x=list(map(float, x)),
                measured=quality(inst, d),
                noiseless=quality(inst, qaoa_probs(inst, np.array(x), p)),
                cvar_measured=cvar(inst, d, ALPHA),
                cvar_noiseless=cvar(inst, qaoa_probs(inst, np.array(x), p), ALPHA),
                probs=d.tolist())
            m = entry["meas"][kind]
            print(f"  n={n:2d} p={p} [{kind:9s}] {n2q:4d} 2q  measured ratio "
                  f"{m['measured']['ratio']:.3f}  (noiseless {m['noiseless']['ratio']:.3f})  "
                  f"feas {m['measured']['feas_mass']:.3f}")
        scan[n]["depths"][p] = entry

    with open(OUT, "w") as fh:
        json.dump(dict(provenance=dict(backend=str(backend.name), sizes=SIZES, depths=DEPTHS,
                                       alpha=ALPHA, shots=SHOTS_FINAL,
                                       eta_law=[ETA_A, ETA_B],
                                       date=time.strftime("%Y-%m-%d %H:%M:%S")),
                       ledger=dict(jobs=n_jobs_used, shots=shots_used,
                                   qpu_seconds=qpu_used, log=job_log),
                       scan=scan), fh)
    print(f"  saved {OUT} through n={n}  ({n_jobs_used} jobs, {qpu_used:.1f} s)")

print(f"\nDONE: {n_jobs_used} jobs, {shots_used:,} shots, {qpu_used:.1f} s metered QPU")

    job   1 [n6p1_noiseless] id=da3rq5rotlns739a2v50   5.93 qpu s (model  5.93)     14 s wall  (total    5.9 / 400 s)
  n= 6 p=1 [noiseless]   45 2q  measured ratio 0.739  (noiseless 0.788)  feas 0.393
    job   2 [n6p2_noiseless] id=da3rq9e1vhnc73fk7oag   5.97 qpu s (model  5.97)     12 s wall  (total   11.9 / 400 s)
  n= 6 p=2 [noiseless]   90 2q  measured ratio 0.847  (noiseless 0.930)  feas 0.531
    job   3 [n6p3_noiseless] id=da3rqcc3jnrc73aff5dg   6.01 qpu s (model  6.01)     23 s wall  (total   17.9 / 400 s)
  n= 6 p=3 [noiseless]  135 2q  measured ratio 0.845  (noiseless 0.952)  feas 0.522
    job   4 [n6p4_noiseless] id=da3rqi43jnrc73aff5k0   6.05 qpu s (model  6.05)     12 s wall  (total   23.9 / 400 s)
  n= 6 p=4 [noiseless]  180 2q  measured ratio 0.796  (noiseless 0.943)  feas 0.452
    job   5 [n6p5_noiseless] id=da3rqle1vhnc73fk7olg   6.09 qpu s (model  6.09)     13 s wall  (total   30.0 / 400 s)
  n= 6 p=5 [noiseless]  225 2q  measured ratio 0.790  (noiseless 0.963)  f

In [7]:
print(f"{'n':>3}{'p':>3}{'2q':>6}{'eta_pred':>10}{'measured':>10}{'noiseless':>11}")
for n in SIZES:
    for p in DEPTHS:
        e = scan[n]["depths"].get(p, {})
        m = e.get("meas", {}).get("noiseless")
        if m:
            print(f"{n:>3}{p:>3}{e['n2q']:>6}{e['eta_pred']:>10.3f}"
                  f"{m['measured']['ratio']:>10.3f}{m['noiseless']['ratio']:>11.3f}")
    best = max((p for p in DEPTHS if scan[n]['depths'].get(p, {}).get('meas', {}).get('noiseless')),
               key=lambda p: scan[n]['depths'][p]['meas']['noiseless']['measured']['ratio'],
               default=None)
    print(f"   -> best measured depth at n={n}: p={best}\n")

  n  p    2q  eta_pred  measured  noiseless
  6  1    45     0.223     0.739      0.788
  6  2    90     0.269     0.847      0.930
  6  3   135     0.312     0.845      0.952
  6  4   180     0.352     0.796      0.943
  6  5   225     0.390     0.790      0.963
   -> best measured depth at n=6: p=2

  8  1    84     0.263     0.659      0.692
  8  2   168     0.342     0.781      0.885
  8  3   252     0.412     0.784      0.932
  8  4   336     0.475     0.772      0.959
  8  5   420     0.531     0.747      0.968
   -> best measured depth at n=8: p=3

 10  1   135     0.312     0.619      0.657
 10  2   270     0.426     0.758      0.902
 10  3   405     0.522     0.742      0.933
 10  4   540     0.602     0.713      0.946
 10  5   675     0.668     0.675      0.961
   -> best measured depth at n=10: p=2

 12  1   198     0.368     0.588      0.609
 12  2   396     0.516     0.727      0.882
 12  3   594     0.630     0.710      0.918
 12  4   792     0.716     0.671      0.932
 1